In [14]:
# Import & Setup 
import pandas as pd
import numpy as np 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pathlib import Path
import joblib

FEATURES_DIR = Path().resolve().parent / 'data' / 'features'
MODELS_DIR = Path().resolve().parent / 'src' / 'models' / 'data'


print('Imports complete!')

Imports complete!


In [15]:
# Load & check data
df = pd.read_csv(FEATURES_DIR / 'model_dataset.csv')

print(df.head())

  Abbreviation  Year  RoundNumber  AvgFinishLast3  AvgFinishLast5  \
0          VER  2023            6           1.333             1.4   
1          ALO  2023            6           3.333             3.2   
2          OCO  2023            6          12.667            12.8   
3          HAM  2023            6           4.667             4.8   
4          RUS  2023            6          10.000             8.2   

   DNFRateLast5  AvgLapTime  AvgLapTimeLast3  AvgLapTimeLast5  AvgPitTime  \
0           0.0      82.814           93.127           94.138      24.484   
1           0.0      82.437           93.590           94.590     150.810   
2           0.4      83.467           94.433           95.535      25.312   
3           0.0      83.481           94.110           95.064      25.554   
4           0.2      83.460           94.707           95.365      30.875   

   ...  Corners  TrackLength  TotalLaps  TrackTemp  AirTemp  Rainfall  \
0  ...       19      2950.03       78.0      39.2

In [16]:
# Data preprocessing
# One-hot encode SafetyCar, VirtualSafetyCar, RedFlag
df[['SafetyCar', 'VirtualSafetyCar', 'RedFlag']] = df[['SafetyCar', 'VirtualSafetyCar', 'RedFlag']].astype(int)

# Drop Abbreviation, RoundNumber & Event 
df = df.drop(columns=['Abbreviation', 'RoundNumber', 'Event'])

# Separate into features & target
X = df[['Year', 'AvgTeamFinishLast3', 'TeamDNFRateLast5', 'Corners', 'TrackLength', 'TotalLaps', 'TrackTemp', 'AirTemp', 'Rainfall', 'SafetyCar', 'VirtualSafetyCar', 'RedFlag']]
y = df['AvgTeamFinishLast5']

print(f'X Features: \n{X}\n\n')
print(f'Target Feature: \n{y}\n')

X Features: 
     Year  AvgTeamFinishLast3  TeamDNFRateLast5  Corners  TrackLength  \
0    2023               1.667               0.0       19      2950.03   
1    2023               7.333               0.0       19      2950.03   
2    2023              10.667               0.2       19      2950.03   
3    2023               6.000               0.2       19      2950.03   
4    2023               4.667               0.0       19      2950.03   
..    ...                 ...               ...      ...          ...   
889  2025               5.333               0.0       16      5040.39   
890  2025               9.000               0.2       16      5040.39   
891  2025              13.667               0.2       16      5040.39   
892  2025              15.000               0.0       16      5040.39   
893  2025              16.333               0.0       16      5040.39   

     TotalLaps  TrackTemp  AirTemp  Rainfall  SafetyCar  VirtualSafetyCar  \
0         78.0      39.26    25.0

In [17]:
# Create train & test split
X_train = X[(X['Year'] == 2023) | (X['Year'] == 2024)]
y_train = y[X_train.index]

X_test = X[(X['Year'] == 2025)]
y_test = y[X_test.index]

# Drop Year to prevent model skew
X_train = X_train.drop(columns=['Year'])
X_test = X_test.drop(columns=['Year'])

In [32]:
# Create RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor

rfr_model = RandomForestRegressor(random_state=42, 
                                  max_depth=50,
                                  n_estimators=10,
                                  min_samples_leaf=10,
                                  min_samples_split=10)
rfr_model.fit(X_train, y_train)

rfr_predictions = rfr_model.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test, rfr_predictions)
mse = mean_squared_error(y_test, rfr_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, rfr_predictions)

print(f"Mean Absolute Error: {mae}")
print(f"Mean Squared Error: {mse}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R-squared Score: {r2}")



Mean Absolute Error: 1.263482882886332
Mean Squared Error: 2.601974675133292
Root Mean Squared Error: 1.6130637542060426
R-squared Score: 0.8389201104159223


In [33]:
# Save trained model to a file
joblib.dump(rfr_model, MODELS_DIR / 'team_consistency_model.pkl')

['E:\\VSCode\\f1-race-forecasting\\src\\models\\data\\team_consistency_model.pkl']